<a href="https://colab.research.google.com/github/gatoaeroespacial/ArquitecturaProject/blob/master/Taller2_Manizales_Transporte_HPC_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Taller #2 – Programación Concurrente y Distribuida  
**Tema:** Gestión del transporte público en Manizales con HPC y Machine Learning  
**Autor(es):** *Juan Mauricio Arias Hernandez*, *Juan Pablo Jimenez Gonzalez*


---

## 0. Objetivo
Construir un pipeline de análisis y modelos de *Machine Learning* para la **gestión del transporte público** en Manizales (rutas, paradas, demanda, tiempos de espera), comparando **ejecución serial vs paralela** en Google Colab (HPC a pequeña escala).

**Entregables clave:**
- Al menos **5 tipos de análisis** (descriptivo, clustering, predicción, reglas de asociación, visualización).
- Uso de **recursos concurrentes/paralelos** (Dask / joblib) y comparación de tiempos.
- **Documentación completa** dentro de este notebook.


## 1. Datos: fuente, estructura y diccionario

> Usaremos dos dataset público relacionado con **Nueva York**.
Los dataset se encuentran en los siguientes link:

1. https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95/about_data

2. https://data.cityofnewyork.us/Transportation/Bus-Breakdown-and-Delays/ez4e-fazm/about_data

El primer dataset cuenta con 2.21M de datos y 29 columnas que son las siguientes:

`CRASH DATE`, `CRASH TIME`, `BOROUGH`, `LOCATION`, `ON STREET NAME`, `NUMBER OF PERSONS INJURED`, `NUMBER OF PERSONS KILLED`, `NUMBER OF PEDESTRIANS INJURED`, `NUMBER OF PEDESTRIANS KILLED`, `NUMBER OF CYCLIST INJURED`, `NUMBER OF CYCLIST KILLED`, `NUMBER OF MOTORIST INJURED`, `NUMBER OF MOTORIST KILLED`, `COLLISION_ID`, `VEHICLE TYPE CODE 1`, `VEHICLE TYPE CODE 2`, `VEHICLE TYPE CODE 3`, `VEHICLE TYPE CODE 4`, `VEHICLE TYPE CODE 5`
`CONTRIBUTING FACTOR VEHICLE 1`, `CONTRIBUTING FACTOR VEHICLE 5`, `CONTRIBUTING FACTOR VEHICLE 4`, `CONTRIBUTING FACTOR VEHICLE 3`, `CONTRIBUTING FACTOR VEHICLE 2`, `ZIP CODE`, `LATITUDE`, `LONGITUDE`, `CROSS STREET NAME`, `OFF STREET NAME`

Sin embargo no se usaron todas ellas, si no que se eliminaros las siguientes porque no tenian razón de ser en el estudio:

`ZIP CODE`, `LATITUDE`, `LONGITUDE`, `CROSS STREET NAME`, `OFF STREET NAME`.





In [ ]:

# === 1.1 Instalación de librerías (ejecuta en Colab) ===
# %pip install -q dask[complete] mlxtend geopandas folium contexttimer

import sys, os, time, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

try:
    import dask.dataframe as dd
    from dask.distributed import Client
    HAS_DASK = True
except Exception:
    HAS_DASK = False

try:
    from mlxtend.frequent_patterns import apriori, association_rules
    HAS_MLXTEND = True
except Exception:
    HAS_MLXTEND = False

try:
    import folium
    HAS_FOLIUM = True
except Exception:
    HAS_FOLIUM = False

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Dask available:", HAS_DASK)
print("mlxtend available:", HAS_MLXTEND)
print("Folium available:", HAS_FOLIUM)


In [ ]:

# === 1.2 Carga de datos ===
DATA_URL = ""  # pega aquí la URL directa al CSV si la tienes

df = None
if DATA_URL:
    try:
        df = pd.read_csv(DATA_URL)
        print("Datos cargados desde URL:", DATA_URL)
    except Exception as e:
        print("No se pudo leer desde la URL. Error:", e)

if df is None:
    try:
        from google.colab import files  # disponible en Colab
        print("Sube un archivo CSV con los datos...")
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        df = pd.read_csv(fname)
        print("Datos cargados desde archivo:", fname)
    except Exception as e:
        raise RuntimeError("No hay datos. Proporciona una URL válida o sube un CSV.") from e

print("Shape:", df.shape)
df.head()


In [ ]:

# === 1.3 Diccionario de datos (inferencia básica) ===
def describe_dataframe(dfx: pd.DataFrame, n_unique_top=5):
    info = []
    for col in dfx.columns:
        dtype = str(dfx[col].dtype)
        n_null = dfx[col].isna().sum()
        n_unique = dfx[col].nunique(dropna=True)
        sample = dfx[col].dropna().unique()[:n_unique_top]
        info.append({
            "columna": col,
            "dtype": dtype,
            "nulos": int(n_null),
            "cardinalidad": int(n_unique),
            "muestra_valores": sample
        })
    dict_df = pd.DataFrame(info)
    return dict_df

dict_datos = describe_dataframe(df)
dict_datos



## 2. Preprocesamiento de datos
- Conversión de fechas/horas
- Limpieza de nulos y duplicados
- Estandarización de nombres de columnas clave (si existen)
- Creación de variables derivadas: hora, día, velocidad media, headway, etc.


In [ ]:

# === 2.1 Normalización de columnas frecuentes ===
df_cols = {c.lower(): c for c in df.columns}

def first_match(cands):
    for c in cands:
        if c in df_cols:
            return df_cols[c]
    return None

col_time = first_match(["timestamp", "fecha_hora", "datetime", "hora", "fecha"])
col_lat  = first_match(["lat", "latitude", "y"])
col_lon  = first_match(["lon", "lng", "longitude", "x"])
col_route= first_match(["route_id", "ruta", "id_ruta"])
col_stop = first_match(["stop_id", "parada", "id_parada"])
col_speed= first_match(["speed", "velocidad"])
col_head = first_match(["headway", "tiempo_espera", "espera"])
col_pass = first_match(["passengers", "pasajeros", "aforo"])
col_veh  = first_match(["vehicle_id", "bus_id", "id_vehiculo"])

standard_cols = {
    "time": col_time, "lat": col_lat, "lon": col_lon,
    "route": col_route, "stop": col_stop, "speed": col_speed,
    "headway": col_head, "passengers": col_pass, "vehicle": col_veh
}
standard_cols


In [ ]:

# === 2.2 Conversión de fecha/hora y features temporales ===
if standard_cols["time"]:
    df["ts"] = pd.to_datetime(df[standard_cols["time"]], errors="coerce", utc=True).dt.tz_convert(None)
    df["hour"] = df["ts"].dt.hour
    df["dow"] = df["ts"].dt.dayofweek
else:
    print("Advertencia: No se encontró columna de tiempo.")
    df["ts"] = pd.NaT
    df["hour"] = np.nan
    df["dow"] = np.nan

df = df.drop_duplicates()
df = df.dropna(how="all")

print("Filas tras limpieza:", len(df))
df.head()



## 3. Análisis Exploratorio (EDA) – Descriptivo
- Estadísticas generales
- Distribuciones por hora
- Top rutas / paradas


In [ ]:

import matplotlib.pyplot as plt
display(df.describe(include="all"))

if "hour" in df:
    counts_by_hour = df.groupby("hour").size()
    plt.figure()
    counts_by_hour.plot(kind="bar")
    plt.title("Eventos por hora")
    plt.xlabel("Hora")
    plt.ylabel("Conteo")
    plt.show()

if standard_cols["route"]:
    top_routes = df[standard_cols["route"]].value_counts().head(10)
    plt.figure()
    top_routes.plot(kind="bar")
    plt.title("Top 10 rutas")
    plt.xlabel("Ruta")
    plt.ylabel("Conteo")
    plt.show()

if standard_cols["stop"]:
    top_stops = df[standard_cols["stop"]].value_counts().head(10)
    plt.figure()
    top_stops.plot(kind="bar")
    plt.title("Top 10 paradas")
    plt.xlabel("Parada")
    plt.ylabel("Conteo")
    plt.show()



### 3.1 Mapa de puntos (si existen lat/lon)


In [ ]:

if HAS_FOLIUM and standard_cols["lat"] and standard_cols["lon"]:
    lat_m = df[standard_cols["lat"]].astype(float).median()
    lon_m = df[standard_cols["lon"]].astype(float).median()
    m = folium.Map(location=[lat_m, lon_m], zoom_start=12)
    sample_map = df[[standard_cols["lat"], standard_cols["lon"]]].dropna().sample(min(3000, len(df)), random_state=42)
    for _, row in sample_map.iterrows():
        folium.CircleMarker(
            location=[float(row[standard_cols["lat"]]), float(row[standard_cols["lon"]])],
            radius=1
        ).add_to(m)
    m
else:
    print("Folium no disponible o faltan columnas lat/lon.")



## 4. Clustering geográfico (K-Means)


In [ ]:

if standard_cols["lat"] and standard_cols["lon"]:
    coords = df[[standard_cols["lat"], standard_cols["lon"]]].dropna().astype(float)
    if len(coords) > 100:
        k = 8
        km = KMeans(n_clusters=k, n_init="auto", random_state=42)
        clusters = km.fit_predict(coords)
        df.loc[coords.index, "cluster"] = clusters
        print(df["cluster"].value_counts().sort_index())
        import matplotlib.pyplot as plt
        plt.figure()
        plt.scatter(coords.iloc[:3000, 0], coords.iloc[:3000, 1], s=2)
        plt.title("Muestra de puntos (sin colorear)")
        plt.xlabel("lat")
        plt.ylabel("lon")
        plt.show()
    else:
        print("Muy pocos puntos para clustering.")
else:
    print("No hay lat/lon para clustering.")



## 5. Predicción del tiempo de espera (Regresión)


In [ ]:

if standard_cols["headway"] and df[standard_cols["headway"]].notna().any():
    df["target_headway"] = pd.to_numeric(df[standard_cols["headway"]], errors="coerce")
else:
    if standard_cols["time"] and (standard_cols["vehicle"] or standard_cols["route"]):
        key = standard_cols["vehicle"] if standard_cols["vehicle"] else standard_cols["route"]
        temp = df.sort_values(["ts", key]).copy()
        temp["target_headway"] = temp.groupby(key)["ts"].diff().dt.total_seconds()
        df["target_headway"] = temp["target_headway"]
    else:
        rng = np.random.default_rng(42)
        df["target_headway"] = rng.normal(300, 60, size=len(df)).clip(30, 1200)

feature_cols = []
if "hour" in df: feature_cols.append("hour")
if "dow" in df: feature_cols.append("dow")
if standard_cols["speed"]: feature_cols.append(standard_cols["speed"])
if standard_cols["route"]: feature_cols.append(standard_cols["route"])
if standard_cols["stop"]:  feature_cols.append(standard_cols["stop"])

X = df[feature_cols].copy()
y = df["target_headway"].astype(float)

for c in feature_cols:
    if X[c].dtype == "object":
        X[c] = X[c].astype("category").cat.codes

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
y = y.replace([np.inf, -np.inf], np.nan).fillna(y.median())

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import time

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
t0 = time.time()
rf.fit(X_train, y_train)
t1 = time.time()

pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2  = r2_score(y_test, pred)

print(f"Tiempo de entrenamiento: {t1 - t0:.2f} s")
print(f"MAE: {mae:.2f}")
print(f"R2:  {r2:.3f}")



## 6. Reglas de asociación (horario–ruta)


In [ ]:

if HAS_MLXTEND and standard_cols["route"]:
    tx = pd.crosstab(df.get("hour"), df[standard_cols["route"]].astype(str))
    tx = tx.loc[:, tx.sum() > max(5, len(tx)*0.02)]
    from mlxtend.frequent_patterns import apriori, association_rules
    freq = apriori(tx.astype(bool), min_support=0.1, use_colnames=True)
    rules = association_rules(freq, metric="lift", min_threshold=1.0).sort_values("lift", ascending=False)
    display(rules.head(15))
else:
    print("mlxtend no disponible o falta columna de ruta; se omite esta sección.")



## 7. Paralelización con Dask (serial vs paralelo)


In [ ]:

def heavy_groupby_pandas(dfx):
    cols = []
    if "dow" in dfx.columns: cols.append("dow")
    if not cols:
        dfx = dfx.copy()
        dfx["dummy"] = 1
        cols = ["dummy"]
    key_speed = standard_cols["speed"] if standard_cols["speed"] else cols[0]
    agg = dfx.groupby(cols).agg(
        n=("ts", "count"),
        mean_speed=(key_speed, "mean")
    )
    return agg

t0 = time.time()
agg_serial = heavy_groupby_pandas(df)
t1 = time.time()
print(f"Tiempo serial (pandas): {t1 - t0:.3f} s")
display(agg_serial.head())

if HAS_DASK:
    try:
        from dask.distributed import Client
        client = Client(processes=True, threads_per_worker=2, n_workers=2, memory_limit="2GB")
        print(client)
        ddf = dd.from_pandas(df, npartitions=8)
        cols = [c for c in ["dow"] if c in ddf.columns]
        if not cols:
            ddf = ddf.assign(dummy=1)
            cols = ["dummy"]
        key_speed = standard_cols["speed"] if standard_cols["speed"] else cols[0]
        t2 = time.time()
        agg_parallel = ddf.groupby(cols).agg({"ts":"count", key_speed:"mean"}).compute()
        t3 = time.time()
        print(f"Tiempo paralelo (Dask): {t3 - t2:.3f} s")
        display(agg_parallel.head())
        client.close()
    except Exception as e:
        print("Fallo Dask:", e)
else:
    print("Dask no disponible.")



## 8. Importancia de variables


In [ ]:

try:
    import matplotlib.pyplot as plt
    importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print(importances.head(10))
    plt.figure()
    importances.head(10).plot(kind="bar")
    plt.title("Top 10 características más importantes")
    plt.xlabel("Feature")
    plt.ylabel("Importancia")
    plt.show()
except Exception as e:
    print("No se pudo mostrar importancia de variables:", e)



## 9. Interpretación y conclusiones
- Descriptivo: patrones por hora y por ruta.
- Clustering: zonas de alta densidad.
- Predicción: desempeño (MAE, R²) y variables relevantes.
- Reglas de asociación: combinaciones destacadas.
- HPC: diferencias de tiempo serial vs paralelo (menciona cifras).

## 10. Trabajo futuro
- Enriquecer con clima, obras viales, eventos.
- Mejor ingeniería de atributos y tuning de hiperparámetros.
- Integración con un simulador de tráfico (SUMO/SimPy) y pipeline en la nube.
